# 定式库 · 云端 GPU 批量 AI 局部评估（KataGo） / Joseki library · cloud GPU batch AI local evaluation (KataGo)

这个笔记本把流水线里最慢的一步（`analyze2.py`，核显上每组约 15 到 30 分钟）搬到带独立显卡的机器上批量运行。分析代码就是本机流水线用的 `pipeline/analyze2.py` 本身（连同 `treesync.py`、`wqutil.py` 一起写进下面的单元格），所以输出的 `g_<名称>.a2.json` 与本机生成的是同一种文件：行棋方视角、带树指纹的 `_meta`、可以断点续跑，`wq.py check` 直接通过，不需要 `wq.py stamp`。

This notebook moves the slowest step of the local pipeline (`analyze2.py`, about 15 to 30 minutes per group on the integrated GPU) onto a machine with a discrete GPU and runs it as a batch. The analysis code is the local pipeline's own `pipeline/analyze2.py` (written into the cells below together with `treesync.py` and `wqutil.py`), so the `g_<name>.a2.json` files are the same kind of file the local run writes: side-to-move perspective, a `_meta` block with the tree fingerprint, resumable, accepted by `wq.py check` as they are, no `wq.py stamp` needed.

**使用步骤 / Steps**

1. 选 GPU 运行时（Colab：*运行时 → 更改运行时类型 → T4 GPU*；Kaggle：右侧 *Session options → Accelerator → GPU*，并把 *Internet* 打开）。
   Select a GPU runtime (Colab: *Runtime → Change runtime type → T4 GPU*; Kaggle: right panel *Session options → Accelerator → GPU*, and switch *Internet* on).
2. 先把两样东西放好：`n_json_<日期>.zip`（变化树）和权重文件 `kata1-tf3-b11c768-s12752M-d6531M.bin.gz`（约 262 MB）。Colab 推荐放进 Google Drive 的 `MyDrive/josekiapp/`；Kaggle 做成私有 Dataset 并添加到笔记本。
   Put two things in place first: `n_json_<date>.zip` (the variation trees) and the weights file `kata1-tf3-b11c768-s12752M-d6531M.bin.gz` (about 262 MB). On Colab put them in `MyDrive/josekiapp/` on Google Drive; on Kaggle make them a private Dataset and add it to the notebook.
3. 从上到下依次运行各单元格。断线后重新连接并重新运行全部单元格：已经完成的分组会被跳过，没完成的分组从检查点继续（前提是输出放在 Drive 或 `/kaggle/working`）。
   Run the cells from top to bottom. After a disconnect reconnect and run all cells again: finished groups are skipped and an unfinished group continues from its checkpoint (as long as the output is on Drive or in `/kaggle/working`).
4. 最后一个单元格生成 `a2_results.zip`。下载后按 README_cloud.md 把里面的 `g_*.a2.json` 放进 `D:\tmp\work`，再运行 `wq.py check`。
   The last cell builds `a2_results.zip`. After downloading, put the `g_*.a2.json` files into `D:\tmp\work` as README_cloud.md describes and run `wq.py check`.

**使用条款（必读） / Terms of use (must read)**

Colab 免费版只能当作交互式笔记本使用。这个笔记本只在笔记本进程内部跑批处理：不开端口，不用 SSH、隧道或远程桌面，不提供对外服务。**不要**加保活脚本、ngrok 或 cloudflared 隧道，也不要把 Colab 当作网页实时连接的服务器；Colab 免费版 FAQ 明确禁止这些做法，账号可能被限流或封禁。请保持浏览器页面打开、人在附近；如果提示已达到使用上限，就停下来，改天再来或换平台，不要想办法绕过。

The Colab free tier may only be used as an interactive notebook. This notebook only runs a batch job inside the notebook process: it opens no ports and uses no SSH, tunnels or remote desktop, and serves nothing. **Do not** add a keep-alive script, an ngrok or cloudflared tunnel, and do not use Colab as a server that a web page connects to in real time; the Colab free-tier FAQ prohibits these and the account may be throttled or banned. Keep the browser tab open and stay nearby; if it says the usage limit has been reached, stop and come back another day or switch platform instead of working around it.

## 1. 环境与设置 / Environment and settings

第一个单元格打印 GPU、磁盘和网络情况。向助手求助时请把这一段输出原样贴回去（里面没有账号或密码）。

The first cell prints the GPU, disk and network. When you ask for help, paste this output back as it is (it holds no account data or passwords).

In [ ]:
import glob, hashlib, json, os, re, shutil, subprocess, sys, time, urllib.request, zipfile

# ---- settings (the defaults are the ones the README describes)
USE_DRIVE = True          # Colab: keep input, progress and results in MyDrive/josekiapp, so a disconnect loses nothing. False: upload by hand (lost on disconnect).
DRIVE_DIR = 'josekiapp'   # folder in My Drive that holds the weights, the n_json zip and (later) the results
WEIGHTS_PATH = ''         # only when the weights are somewhere unusual: the full path of the .bin.gz
KATAGO_BUILD = 'cuda12.1-cudnn9.8.0'   # if libraries are missing try cuda12.5-cudnn9.8.0, then cuda12.8-cudnn9.8.0
WEIGHTS_NAME = 'kata1-tf3-b11c768-s12752M-d6531M.bin.gz'
EXPECTED_SHA256 = 'c470f51078d31051961dfc372aaa3948a3eec0b93ed52075d16f55a2e17b8cec'   # the weights the local pipeline uses; '' skips the check, but then old and new analysis are not comparable
EXPECTED_BYTES = 262040299
VISITS, FORCED = 220, 100  # the local baseline: do not change, or old and new results are no longer comparable

ON_KAGGLE = os.path.exists('/kaggle/working')
ON_COLAB = (not ON_KAGGLE) and ('google.colab' in sys.modules or os.path.exists('/content'))
WORK = '/kaggle/working' if ON_KAGGLE else ('/content' if ON_COLAB else os.getcwd())
os.chdir(WORK)   # the scripts of the next cells are written here and run from here
if WORK not in sys.path:
    sys.path.insert(0, WORK)


def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()


def run_stream(cmd, env=None, every=30):
    '''Run a command and show its output as it comes (same on Colab and Kaggle). Progress lines are thinned out to one per `every` seconds.'''
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace', env=env, bufsize=1)
    last = 0.0
    for line in p.stdout:
        if re.match(r'\d\d:\d\d:\d\d phase ', line) and time.time() - last < every and 'queries' in line and not re.search(r'(\d+)/\1 queries', line):
            continue
        if re.match(r'\d\d:\d\d:\d\d phase ', line):
            last = time.time()
        print(line, end='', flush=True)
    return p.wait()


try:
    urllib.request.urlopen('https://github.com', timeout=10)
    INTERNET = 'yes'
except Exception as e:
    INTERNET = 'NO (%s)' % e
GPU = sh('nvidia-smi')
print('=== paste this block back when you ask for help ===')
print('platform :', 'Kaggle' if ON_KAGGLE else 'Colab' if ON_COLAB else 'other', '| python', sys.version.split()[0], '| work dir', WORK)
print('internet to github.com:', INTERNET)
print(GPU or '!! nvidia-smi printed nothing: no GPU runtime')
print(sh('df -h ' + WORK))
print(sh('free -h | head -2'))
print('cpu cores:', os.cpu_count())
print('=== end of block ===')
assert GPU, 'No GPU: select a GPU runtime first (Colab: Runtime > Change runtime type > T4 GPU; Kaggle: Session options > Accelerator).'
if ON_KAGGLE:
    assert INTERNET == 'yes', 'Internet is off: switch it on in the right panel (Session options > Internet). KataGo is downloaded from GitHub.'

下面五个单元格把脚本写进工作目录，内容与仓库里的同名文件一致（由 `remote/make_notebook.py` 生成，不要手工改）。

The next five cells write the scripts into the working directory. Their text is identical to the files of the same name in the repository (generated by `remote/make_notebook.py`, do not edit by hand).

In [ ]:
%%writefile treesync.py
"""Keep derived files tied to the variation tree they were made from.

Incident: a2.json (KataGo analysis) and kp_*.json (key point notes) are keyed by
node id. When another video of a group finished downloading, build_tree produced a
different tree, the ids shifted, and the old files were silently merged into the
wrong nodes. Now every derived file carries a fingerprint of the tree it belongs to
(inside a "_meta" key, so old readers that look up node ids are not affected) and
the loaders compare it before anything is merged.
"""
import hashlib
import json
import re
import time

SCHEMA = 2
PERSP = 'stm'          # a2.json scores and winrates are from the side to move (never Black)
USABLE = ('ok', 'legacy')   # analysis statuses that may be merged into an SGF
L = 'ABCDEFGHJKLMNOPQRST'
COLS = 18              # the recognised board is 18 columns wide, see board.py


def tree_fingerprint(N):
    """Hash of the tree structure: ids, parents, moves, children, flags and positions.

    Video time spans (the 'v' field) are left out on purpose: a timing shift cannot make
    an analysis wrong, and including them would force a GPU re-run for nothing. Any change
    of node ids, moves or positions does change the fingerprint.
    """
    canon = [[n['id'], n['par'], n['mv'], n['ch'], bool(n['dead']), bool(n['setup']), n['pos']] for n in N]
    return hashlib.sha1(json.dumps(canon, separators=(',', ':')).encode('ascii')).hexdigest()[:16]


def pos_key(n):
    """Identity of a node that survives a rebuild of the tree: the position plus who moved last."""
    return hashlib.sha1(('%s|%s' % (n['pos'], n['mv'][1] if n['mv'] else '')).encode('ascii')).hexdigest()[:12]


def pos_keys(N):
    """{node id as str: position key} for all live nodes."""
    return {str(n['id']): pos_key(n) for n in N if not n['dead']}


# ---------------------------------------------------------------- analysis (a2.json)

def make_meta(N, **extra):
    """The _meta block stored in a2.json. persp records that scores are side-to-move."""
    m = {'schema': SCHEMA, 'persp': PERSP, 'tree_fp': tree_fingerprint(N), 'n_nodes': len(N),
         'complete': True, 'created': time.strftime('%Y-%m-%d %H:%M:%S')}
    m.update(extra)
    return m


def _entries(A):
    """(key, entry) for real analysis entries; skips _meta and error stubs."""
    for k, a in A.items():
        if k != '_meta' and isinstance(a, dict) and 'cand' in a:
            yield k, a


def _idx(m):
    """GTP move text ('D4') to the pipeline's point index (row * 18 + col), or None if not on the 18x19 board."""
    mt = re.fullmatch(r'([A-HJ-T])(\d{1,2})', m or '')
    if not mt or L.index(mt.group(1)) >= COLS or not 1 <= int(mt.group(2)) <= 19:
        return None
    return (19 - int(mt.group(2))) * COLS + L.index(mt.group(1))


def perspective_suspect(A, min_nodes=5):
    """Detect scores that are not side-to-move (incident 1: Black perspective used as if it were stm).

    KataGo orders candidates by how good they are for the side to move. With correct
    side-to-move scores the first candidate has (nearly) the best score for both colours.
    With Black-perspective scores that holds only for Black; for White the first candidate has
    the WORST score. Calibrated on real data: correct files give ~100% for White nodes, the old
    Black-perspective files gave 13% to 22%. Returns (suspect, {turn: (ok, total)}).
    """
    seen = {'B': [0, 0], 'W': [0, 0]}
    for k, a in _entries(A):
        if not re.fullmatch(r'\d+(_\d+)?', k) or a.get('turn') not in seen:   # node ids, also the context-free '<id>_0' queries
            continue
        c = [m for m in a['cand'] if not m.get('f')]   # forced evaluations of teacher moves are not ordered
        if len(c) < 3:
            continue
        seen[a['turn']][1] += 1
        seen[a['turn']][0] += c[0]['s'] >= max(m['s'] for m in c) - 1.0
    bad = [t for t, (ok, n) in seen.items() if n >= min_nodes and ok < 0.5 * n]
    return bool(bad), {t: tuple(v) for t, v in seen.items()}


def consistent_with_tree(N, A):
    """Cheap structural check for analysis files that carry no fingerprint (made before this existed).

    Real analysis of this tree: every id exists and is alive, the side to move is the opposite of the
    last mover, and no candidate sits on an occupied point. A file for another tree breaks these quickly.
    Returns (ok, reason).
    """
    n_chk = bad_id = bad_turn = bad_occ = 0
    for k, a in _entries(A):
        m = re.fullmatch(r'(\d+)(_0)?', k)
        if not m:
            continue
        n_chk += 1
        i = int(m.group(1))
        if i >= len(N) or N[i]['dead']:
            bad_id += 1
            continue
        if not m.group(2) and N[i]['mv'] and a.get('turn') == N[i]['mv'][1]:
            bad_turn += 1   # (the _0 entries drop outside moves, so their turn can differ legitimately)
        occ = {int(t[1:]) for t in re.findall(r'[BW]\d+', N[i]['pos'])}
        if any(_idx(c['m']) in occ for c in a['cand']):
            bad_occ += 1
    if not n_chk:
        return False, 'no analysis entries'
    if bad_id > 0.02 * n_chk or bad_turn > 0.05 * n_chk or bad_occ > 0.02 * n_chk:
        return False, 'entries do not fit this tree (unknown ids %d, wrong side to move %d, candidate on a stone %d, of %d)' % (
            bad_id, bad_turn, bad_occ, n_chk)
    return True, ''


def check_analysis(N, A):
    """Decide whether analysis A may be merged for tree N. Returns (status, message).

    ok          fingerprint matches
    legacy      no fingerprint, but it passes the structural check (use wq.py stamp to bless it)
    stale       made for another tree: must be redone
    incomplete  a LIMITed or aborted run
    bad_persp   scores look like they are not side to move
    missing     nothing to merge
    """
    if not isinstance(A, dict) or not any(True for _ in _entries(A)):
        return 'missing', 'no analysis data'
    meta = A.get('_meta')
    sus, stats = perspective_suspect(A)
    if isinstance(meta, dict):
        if meta.get('persp') != PERSP:
            return 'bad_persp', 'a2 meta says perspective %r, expected %r' % (meta.get('persp'), PERSP)
        if meta.get('complete') is False:
            return 'incomplete', 'analysis run was limited or aborted'
        if meta.get('tree_fp') != tree_fingerprint(N):
            return 'stale', 'tree changed since the analysis (a2 was made for %s nodes, tree has %d)' % (meta.get('n_nodes', '?'), len(N))
        if sus:
            return 'bad_persp', 'scores look like Black perspective, first candidate is not the best one: %s' % stats
        return 'ok', ''
    if sus:
        return 'bad_persp', 'no meta and scores look like Black perspective: %s' % stats
    ok, why = consistent_with_tree(N, A)
    if not ok:
        return 'stale', 'no meta and ' + why
    return 'legacy', 'no fingerprint in a2.json (run wq.py stamp to record it)'


# ---------------------------------------------------------------- key point notes (kp_*.json)

def kp_ident(N):
    """Identity block stored in kp files and in the kpin sidecar."""
    return {'tree_fp': tree_fingerprint(N), 'keys': pos_keys(N)}


def check_kp(N, KP, ident=None, strict=False):
    """Pick the notes of KP that belong to tree N. Returns (notes {id: text}, status, message, stamp).

    The notes are written by a separate helper that only saw one version of the tree, and a kp file
    is keyed by node id, so a changed tree must not be merged blindly.

    ident  identity to assume when the file has no _meta (taken from the kpin dump the notes were made from)
    stamp  identity the caller should write into the file as _meta, or None
    status ok / remapped (carried over by position) / legacy (unverified) / stale / missing
    """
    if not isinstance(KP, dict) or not KP:
        return {}, 'missing', 'no kp data', None
    notes = {k: v for k, v in KP.items() if k != '_meta' and isinstance(v, str) and v.strip()}
    meta = KP.get('_meta') if isinstance(KP.get('_meta'), dict) else None
    stamp = None
    if meta is None and ident:
        meta = stamp = ident
    fp = tree_fingerprint(N)
    if meta is None:
        alive = {k: v for k, v in notes.items() if k.isdigit() and int(k) < len(N) and not N[int(k)]['dead']}
        if strict or len(alive) < 0.8 * len(notes):
            return {}, 'stale', 'kp has no fingerprint and %d of %d ids do not fit the tree' % (len(notes) - len(alive), len(notes)), None
        return alive, 'legacy', 'kp has no fingerprint, %d notes applied unverified (run wq.py stamp)' % len(alive), None
    if meta.get('tree_fp') == fp:
        return notes, 'ok', '', stamp
    keys = meta.get('keys')
    if isinstance(keys, dict) and keys:
        now = {pk: k for k, pk in pos_keys(N).items()}
        out = {now[keys[k]]: v for k, v in notes.items() if keys.get(k) in now}
        if out:
            return out, 'remapped', 'tree changed: %d of %d notes carried over by position, the rest need new notes' % (len(out), len(notes)), stamp
    return {}, 'stale', 'tree changed and no note could be matched by position', stamp

In [ ]:
%%writefile wqutil.py
"""Shared helpers for the pipeline scripts: atomic file writes, tolerant json reading, utf-8 and logging.

Why this exists: the pipeline gets killed and restarted a lot, and other tools
(build.ps1, the viewer, a second regen run) read its output while it is being
written. A half written json or sgf must never be visible, so every output goes
through atomic_write_*. Reads go through read_json, which copes with a BOM and
gives an error message that names the file. setup_stdio/utf8_env/Log keep every
log line and child process on utf-8 (Windows consoles default to cp936).
"""
import glob
import json
import os
import random
import shutil
import sys
import threading
import time

_MISSING = object()


def atomic_write_bytes(path, data, tries=8):
    """Write data so readers see either the old file or the new one, never a partial file.

    The data goes to a temp file next to the target, is fsynced, and is then moved
    over the target with os.replace (atomic on the same volume). On Windows
    os.replace can raise PermissionError while another process has the target open
    (build.ps1, a virus scanner, a file server), so it is retried a few times.
    """
    path = os.fspath(path)
    tmp = '%s.tmp.%d.%d' % (path, os.getpid(), threading.get_ident())
    try:
        with open(tmp, 'wb') as f:
            f.write(data)
            f.flush()
            os.fsync(f.fileno())
        for k in range(tries):
            try:
                os.replace(tmp, path)
                return
            except PermissionError:
                if k == tries - 1:
                    raise
                time.sleep(0.25 * (k + 1))
    finally:
        # Only still there when the write or the replace failed: do not leave junk behind.
        if os.path.exists(tmp):
            try:
                os.remove(tmp)
            except OSError:
                pass


def atomic_write_text(path, text, encoding='utf-8'):
    """Atomic text write. Always utf-8 and no newline translation (LF stays LF on Windows)."""
    atomic_write_bytes(path, text.encode(encoding))


def atomic_write_json(path, obj, ensure_ascii=True, **kw):
    """Atomic json write. The object is serialised first, so a TypeError cannot touch the old file."""
    atomic_write_text(path, json.dumps(obj, ensure_ascii=ensure_ascii, **kw))


def read_json(path, default=_MISSING, quarantine=False):
    """Read a json file as utf-8 (a BOM is fine: PowerShell's Set-Content writes one).

    A missing file or a file that does not parse returns `default` when one is given,
    otherwise raises with the path in the message. With quarantine=True an unparsable
    file is renamed to <path>.corrupt so the stage that produces it runs again instead
    of failing on the same broken file forever (typical after a kill during a write).
    """
    try:
        with open(path, encoding='utf-8-sig') as f:
            return json.load(f)
    except FileNotFoundError:
        if default is not _MISSING:
            return default
        raise
    except (ValueError, UnicodeDecodeError) as e:
        if quarantine:
            try:
                os.replace(path, os.fspath(path) + '.corrupt')
            except OSError:
                pass
        if default is not _MISSING:
            return default
        raise ValueError('%s is not valid json: %s' % (path, e)) from e


def find_ffmpeg():
    """Return the ffmpeg path: env FFMPEG, the WinGet install this pipeline was built on, or PATH.

    Returns None when nothing is found; callers must turn that into a clear error
    (the old code did glob(...)[0] at import time and died with an IndexError).
    """
    p = os.environ.get('FFMPEG')
    if p and os.path.exists(p):
        return p
    la = os.environ.get('LOCALAPPDATA')
    if la:
        pat = os.path.join(la, 'Microsoft', 'WinGet', 'Packages', 'Gyan.FFmpeg*', '**', 'ffmpeg.exe')
        hits = glob.glob(pat, recursive=True)
        if hits:
            return hits[0]
    return shutil.which('ffmpeg')


def setup_stdio():
    """Force utf-8 on stdout/stderr.

    A cp936 (GBK) Windows console either garbles Chinese log lines or raises
    UnicodeEncodeError on characters it cannot show. errors='replace' keeps the
    process alive in every case. Child processes get the same via utf8_env().
    """
    for s in (sys.stdout, sys.stderr):
        try:
            s.reconfigure(encoding='utf-8', errors='replace')
        except (AttributeError, ValueError, OSError):
            pass  # not a real text stream (pytest capture, pythonw): nothing to fix


def utf8_env(extra=None):
    """Environment for child processes: utf-8 text mode and unbuffered output (so logs show progress live)."""
    env = dict(os.environ)
    env.update(PYTHONUTF8='1', PYTHONIOENCODING='utf-8', PYTHONUNBUFFERED='1')
    if extra:
        env.update(extra)
    return env


class Log:
    """print() plus an append-only utf-8 log file. Never raises: a broken console must not kill a worker thread."""

    def __init__(self, path_fn):
        self.path_fn = path_fn  # a function, so the log dir can still change after import (tests)
        self.lock = threading.Lock()

    def __call__(self, *a):
        line = time.strftime('%H:%M:%S ') + ' '.join(str(x) for x in a)
        try:
            print(line, flush=True)
        except (UnicodeError, OSError):
            try:
                print(line.encode('ascii', 'replace').decode('ascii'), flush=True)
            except Exception:
                pass
        try:
            p = self.path_fn()
            os.makedirs(os.path.dirname(p), exist_ok=True)
            with self.lock, open(p, 'a', encoding='utf-8') as f:
                f.write(line + '\n')
        except OSError:
            pass


def retry_with_backoff(attempt, tries=4, base=60.0, factor=3.0, cap=900.0, jitter=0.2, sleep=time.sleep, log=lambda *a: None):
    """Call attempt() -> (ok, info) until it succeeds or `tries` are used up. Returns (ok, info of the last try).

    The wait grows base, base*factor, ... up to `cap`, plus some jitter. HTTP 567/412 throttling needs minutes
    to clear, so hammering every 45 s (the old loop) only extends it. The caller must not hold any shared
    lock while this sleeps: the sleep happens here, outside attempt().
    """
    info = ''
    for k in range(tries):
        ok, info = attempt()
        if ok:
            return True, info
        if k < tries - 1:
            wait = min(cap, base * factor ** k) * (1 + random.uniform(0, jitter))
            log('try %d/%d failed (%s), waiting %ds' % (k + 1, tries, info, wait))
            sleep(wait)
    return False, info


class Manifest:
    """Small json file {key: record} for things that failed (videos). Thread safe, atomic, utf-8."""

    def __init__(self, path_fn):
        self.path_fn = path_fn
        self.lock = threading.Lock()

    def _load(self):
        d = read_json(self.path_fn(), default={}, quarantine=True)
        return d if isinstance(d, dict) else {}

    def record(self, key, **fields):
        with self.lock:
            d = self._load()
            old = d.get(str(key), {})
            fields['attempts'] = old.get('attempts', 0) + 1
            fields['time'] = time.strftime('%Y-%m-%d %H:%M:%S')
            d[str(key)] = fields
            os.makedirs(os.path.dirname(self.path_fn()), exist_ok=True)
            atomic_write_json(self.path_fn(), d, ensure_ascii=False, indent=1)

    def clear(self, key):
        with self.lock:
            d = self._load()
            if str(key) in d:
                del d[str(key)]
                atomic_write_json(self.path_fn(), d, ensure_ascii=False, indent=1)

    def all(self):
        with self.lock:
            return self._load()


def single_instance(path):
    """Take an exclusive lock on `path` for the life of the process. Returns the open file (keep it referenced) or None if
    another process already holds it. The OS drops the lock when the process dies, so a crash never leaves a stale lock
    behind. Two pipelines at once would fight over downloads, the GPU and the same output files."""
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    f = open(path, 'a+b')
    try:
        f.seek(0)
        if os.name == 'nt':
            import msvcrt
            msvcrt.locking(f.fileno(), msvcrt.LK_NBLCK, 1)
        else:
            import fcntl
            fcntl.flock(f, fcntl.LOCK_EX | fcntl.LOCK_NB)
    except OSError:
        f.close()
        return None
    return f

In [ ]:
%%writefile analyze2.py
"""KataGo local analysis of every branch point of a variation tree.

Usage: analyze2.py n.json out.json [main_visits=220] [teacher_move_visits=100]     (env LIMIT=n: only the first n nodes)

Reads the tree, asks KataGo (JSON analysis engine) about each branch point inside the region the
lesson is about, evaluates the teacher's moves separately, and writes out.json:
  { "<node id>": {turn, score, wr, cand:[...]}, "<id>_0": same without stones outside the region, "_meta": {...} }

Scores are ALWAYS stored from the side to move (stm). KataGo reports them from Black's point of view
when reportAnalysisWinratesAs=BLACK, so convert() flips them for White. How the engine reports is not
assumed: a handicap position with White to move is asked first (detect_report_as).

Robustness: results are checkpointed to out.json.part (resume after a crash, same tree only), progress with an
ETA goes to out.json.progress.log, a silent or dead engine ends the run with a clear message instead of a hang.
"""
import json
import os
import queue
import re
import subprocess
import sys
import threading
import time

from treesync import make_meta, perspective_suspect, tree_fingerprint
from wqutil import atomic_write_json, read_json, setup_stdio, utf8_env

L = 'ABCDEFGHJKLMNOPQRST'
C = 18  # columns of the recognised board (see board.py)


# ---------------------------------------------------------------- configuration (env overrides, defaults = the original install)

def cfg():
    root = os.environ.get('WEIQI_KATAGO_ROOT', r'D:\workspace\fun\Weiqi\Weiqi_AI')
    E = os.environ.get('WEIQI_KATAGO_DIR', os.path.join(root, 'engine', 'katago-v1.18.0-opencl-windows-x64'))
    return {
        'dir': E,
        'config': os.path.join(E, 'analysis.cfg'),
        'model': os.environ.get('WEIQI_KATAGO_MODEL', os.path.join(root, 'weights', 'network_file', 'kata1-tf3-b11c768-s12752M-d6531M.bin.gz')),
        'window': int(os.environ.get('WEIQI_WINDOW', '8')),            # queries in flight at once (never write everything up front: pipe deadlock risk)
        'stall': float(os.environ.get('WEIQI_STALL_S', '900')),        # seconds without any answer before we give up
    }


def engine_cmd(c):
    if os.environ.get('WEIQI_KATAGO_CMD'):          # a json list; used by the tests to plug in a fake engine
        return json.loads(os.environ['WEIQI_KATAGO_CMD'])
    return [os.path.join(c['dir'], 'katago.exe'), 'analysis', '-model', c['model'], '-config', c['config']]


def cfg_report_as(path):
    """reportAnalysisWinratesAs from the engine config file, or None."""
    try:
        with open(path, encoding='utf-8-sig') as f:
            for line in f:
                m = re.match(r'\s*reportAnalysisWinratesAs\s*=\s*(\w+)', line)
                if m:
                    return m.group(1).upper()
    except OSError:
        pass
    return None


# ---------------------------------------------------------------- perspective (incident 1: Black scores used as side-to-move)

def stm_sign(turn, report_as):
    """Factor that turns a score lead reported by KataGo into the side-to-move value."""
    if report_as == 'SIDETOMOVE':
        return 1
    if report_as == 'BLACK':
        return 1 if turn == 'B' else -1
    if report_as == 'WHITE':
        return 1 if turn == 'W' else -1
    raise ValueError('unknown reportAnalysisWinratesAs %r' % report_as)


def stm_winrate(wr, turn, report_as):
    return wr if stm_sign(turn, report_as) == 1 else 1 - wr


def convert(d, report_as):
    """One KataGo answer -> our record, everything from the side to move. The ONLY place that flips signs."""
    r = d['rootInfo']
    t = r['currentPlayer']
    sg = stm_sign(t, report_as)
    mi = sorted(d['moveInfos'], key=lambda m: m['order'])
    return {'turn': t, 'score': round(sg * r['scoreLead'], 1), 'wr': round(stm_winrate(r['winrate'], t, report_as), 3),
            'cand': [{'m': m['move'], 'v': m['visits'], 's': round(sg * m['scoreLead'], 1),
                      'wr': round(stm_winrate(m['winrate'], t, report_as), 3), 'p': round(m['prior'], 3), 'pv': m['pv'][:8]}
                     for m in mi[:14]]}


def selftest_query():
    """Nine handicap stones, White to move: Black is far ahead, so who the numbers are 'for' shows immediately."""
    stars = [['B', p] for p in ('D4', 'D10', 'D16', 'K4', 'K10', 'K16', 'Q4', 'Q10', 'Q16')]
    return {'id': 'selftest', 'initialStones': stars, 'moves': [], 'initialPlayer': 'W', 'rules': 'japanese', 'komi': 0.5,
            'boardXSize': 19, 'boardYSize': 19, 'maxVisits': 16, 'analyzeTurns': [0]}


def detect_report_as(eng, cfg_file, log):
    """How does this engine report winrates? Returns (BLACK|SIDETOMOVE|WHITE, source).

    Order: env WEIQI_REPORT_AS, a live check with the handicap position, the config file, else BLACK (the original setup).
    A live check beats the config file because the file is only what we hope was loaded.
    """
    if os.environ.get('WEIQI_REPORT_AS'):
        return os.environ['WEIQI_REPORT_AS'].upper(), 'env'
    from_cfg = cfg_report_as(cfg_file)
    try:
        eng.send(selftest_query())
        r = eng.get_for('selftest', 120)['rootInfo']
        if r['currentPlayer'] == 'W':
            found = 'BLACK' if r['winrate'] > 0.65 else ('SIDETOMOVE' if r['winrate'] < 0.35 else None)
            if found:
                if from_cfg and from_cfg != found:
                    log('WARN the engine reports %s but %s says %s: using what the engine really does' % (found, cfg_file, from_cfg))
                return found, 'selftest'
    except (KeyError, TypeError) as e:
        log('WARN perspective self test gave no usable answer (%s), falling back' % e)
    if from_cfg:
        return from_cfg, 'cfg'
    log('WARN could not determine the score perspective, assuming BLACK like the original config')
    return 'BLACK', 'assumed'


# ---------------------------------------------------------------- the engine process

class EngineError(RuntimeError):
    pass


class Engine:
    """KataGo analysis engine behind a reader thread, so waiting for an answer can time out."""

    def __init__(self, cmd, errfile, stall=900):
        self.p = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=errfile, text=True,
                                  encoding='utf-8', env=utf8_env())
        self.q = queue.Queue()
        self.stall = stall
        threading.Thread(target=self._read, daemon=True).start()

    def _read(self):
        for line in self.p.stdout:
            self.q.put(line)
        self.q.put(None)  # EOF: the engine is gone

    def send(self, req):
        try:
            self.p.stdin.write(json.dumps(req) + '\n')
            self.p.stdin.flush()
        except (OSError, ValueError) as e:
            raise EngineError('engine is not accepting input (%s); exit code %s' % (e, self.p.poll()))

    def get(self, timeout=None):
        """Next answer line as dict. Raises EngineError when the engine died or stays silent."""
        wait = timeout or self.stall
        try:
            line = self.q.get(timeout=wait)
        except queue.Empty:
            raise EngineError('no answer from KataGo for %d s (stalled?)' % wait)
        if line is None:
            self.q.put(None)
            raise EngineError('KataGo exited (code %s)' % self.p.poll())
        try:
            return json.loads(line)
        except ValueError:
            return {'_raw': line.strip()[:200]}

    def get_for(self, qid, timeout):
        while True:
            d = self.get(timeout)
            if d.get('id') == qid:
                return d

    def close(self):
        try:
            self.p.stdin.close()
        except OSError:
            pass
        try:
            self.p.wait(timeout=10)
        except subprocess.TimeoutExpired:
            self.p.kill()


# ---------------------------------------------------------------- tree, region, queries

def region(N):
    """Rectangle that covers the lesson: 3%..97% quantiles of all move points +2 lines, snapped to the edges. None if no moves."""
    cs = set()
    for n in N:
        if n['dead']:
            continue
        if n['mv'] is not None:
            cs.add((n['mv'][0] % C, n['mv'][0] // C))
        elif n['par'] == 0:
            for t in re.findall(r'[BW]\d+', n['pos']):
                k = int(t[1:])
                cs.add((k % C, k // C))
    if not cs:
        return None
    xs = sorted(c for c, _ in cs)
    ys = sorted(r for _, r in cs)
    pc = lambda a, p: a[min(len(a) - 1, int(len(a) * p))]
    x0 = max(0, pc(xs, .03) - 2)
    x1 = min(17, pc(xs, .97) + 2)
    y0 = max(0, pc(ys, .03) - 2)
    y1 = min(18, pc(ys, .97) + 2)
    if x0 <= 3:
        x0 = 0
    if y1 >= 15:
        y1 = 18
    if x1 >= 14:
        x1 = 17
    if y0 <= 3:
        y0 = 0
    return dict(x0=x0, x1=x1, y0=y0, y1=y1)


def gtp(c, r):
    return L[c] + str(19 - r)


def parse_pos(pos):
    d = {}
    for t in re.findall(r'[BW]\d+', pos):
        k = int(t[1:])
        d[(k % C, k // C)] = t[0]
    return d


def inreg(R, c, r):
    return R['x0'] <= c <= R['x1'] and R['y0'] <= r <= R['y1']


def path_of(N, i):
    p = []
    while N[i]['mv'] is not None:
        p.append(i)
        i = N[i]['par']
    return i, p[::-1]


def stones_at(N, i):
    """All stones at node i (setup position + moves): decides whether anything lies outside the region."""
    ri, p = path_of(N, i)
    d = dict(parse_pos(N[ri]['pos']))
    for j in p:
        d[(N[j]['mv'][0] % C, N[j]['mv'][0] // C)] = N[j]['mv'][1]
    return d


def query(N, R, allow, i, ctx, qid, visits):
    """KataGo request for node i. ctx=False drops the stones outside the region (the 'without outside influence' control)."""
    ri, p = path_of(N, i)
    ini = [[v, gtp(*k)] for k, v in parse_pos(N[ri]['pos']).items() if ctx or inreg(R, *k)]
    mv = [[N[j]['mv'][1], gtp(N[j]['mv'][0] % C, N[j]['mv'][0] // C)] for j in p
          if (ctx or inreg(R, N[j]['mv'][0] % C, N[j]['mv'][0] // C))]
    return {'id': qid, 'initialStones': ini, 'moves': mv, 'rules': 'japanese', 'komi': 6.5, 'boardXSize': 19, 'boardYSize': 19,
            'analyzeTurns': [len(mv)], 'maxVisits': visits,
            'allowMoves': [{'player': 'B', 'moves': allow, 'untilDepth': 40}, {'player': 'W', 'moves': allow, 'untilDepth': 40}]}


def live_children(N, i):
    return [c for c in N[i]['ch'] if not N[c]['dead']]


def branch_nodes(N):
    """Nodes worth analysing: branch points (leaves and single chains mean nothing) and the first node after a setup."""
    todo = [i for i in range(len(N)) if not N[i]['dead'] and len(live_children(N, i)) >= 2]
    rootkids = [i for i in range(len(N)) if not N[i]['dead'] and N[i]['setup'] and len(live_children(N, i)) == 1 and N[i]['par'] == 0]
    return sorted(set(todo + rootkids))


# ---------------------------------------------------------------- progress and checkpoint

class Progress:
    """Progress lines with counts and ETA: <out>.progress.log (append), <out>.progress.json (latest) and stderr."""

    def __init__(self, out):
        self.log_path, self.json_path = out + '.progress.log', out + '.progress.json'
        self.phase, self.total, self.base, self.nodes_total = '', 0, 0, 0
        self.t0, self.last = time.time(), 0.0

    def start(self, phase, total, already, nodes_total, nodes_done):
        self.phase, self.total, self.base, self.nodes_total = phase, total, already, nodes_total
        self.t0, self.last = time.time(), 0.0
        self.tick(already, nodes_done, force=True)

    def tick(self, done, nodes_done, force=False):
        now = time.time()
        if not force and now - self.last < 5:
            return
        self.last = now
        el = now - self.t0
        new = done - self.base  # answered in this run (resumed ones cost no time)
        eta = (self.total - done) * el / new if new > 0 else None
        line = '%s %s: %d/%d queries, nodes %d/%d, elapsed %ds, eta %s' % (
            time.strftime('%H:%M:%S'), self.phase, done, self.total, nodes_done, self.nodes_total, el,
            ('%ds' % eta) if eta is not None else 'n/a')
        print(line, file=sys.stderr, flush=True)
        try:
            with open(self.log_path, 'a', encoding='utf-8') as f:
                f.write(line + '\n')
            atomic_write_json(self.json_path, {'phase': self.phase, 'done': done, 'total': self.total, 'nodes_done': nodes_done,
                                               'nodes_total': self.nodes_total, 'elapsed_s': round(el),
                                               'eta_s': None if eta is None else round(eta), 'updated': time.strftime('%Y-%m-%d %H:%M:%S')})
        except OSError:
            pass


def load_part(path, fp, visits, report_as):
    """Results of an interrupted run, only if they belong to this tree, these visits and this score perspective."""
    d = read_json(path, default=None, quarantine=True)
    if isinstance(d, dict) and d.get('tree_fp') == fp and d.get('visits') == visits and d.get('report_as') == report_as:
        return d.get('done', {})
    return {}


# ---------------------------------------------------------------- the run

def run_queries(eng, reqs, done, convert_fn, save, prog, window, node_of, every=25):
    """Send the reqs that are not in `done` yet, at most `window` in flight, and store the converted answers in `done`."""
    it = iter([r for r in reqs if r['id'] not in done])
    inflight = set()
    got = base = sum(1 for r in reqs if r['id'] in done)
    nodes = {node_of(r['id']) for r in reqs if r['id'] in done}

    def feed():
        while len(inflight) < window:
            r = next(it, None)
            if r is None:
                return
            eng.send(r)
            inflight.add(r['id'])

    feed()
    since, last_save = 0, time.time()
    while inflight:
        d = eng.get()
        qid = d.get('id')
        if qid not in inflight:
            print('engine: ignoring %s' % json.dumps(d, ensure_ascii=False)[:160], file=sys.stderr)  # a warning line or a stray answer
            continue
        inflight.discard(qid)
        done[qid] = convert_fn(d) if 'moveInfos' in d else {'err': json.dumps(d, ensure_ascii=False)[:120]}
        got += 1
        nodes.add(node_of(qid))
        since += 1
        if since >= every or time.time() - last_save > 30:
            save()
            since, last_save = 0, time.time()
        prog.tick(got, len(nodes))
        feed()
    save()
    prog.tick(got, len(nodes), force=True)
    return got - base


def analyze(N, out, VB, VF, limit=0, log=lambda s: print(s, file=sys.stderr, flush=True)):
    """Whole run. Exit code: 0 ok, 2 nothing to analyse, 3 suspicious scores written, 4 engine failed, 5 engine missing."""
    c = cfg()
    R = region(N)
    if R is None:
        log('ERROR the tree has no moves: nothing to analyse (did board recognition fail for every video?)')
        return 2
    allow = [gtp(x, y) for x in range(R['x0'], R['x1'] + 1) for y in range(R['y0'], R['y1'] + 1)]
    todo = branch_nodes(N)
    if limit:
        todo = todo[:limit]
    fp = tree_fingerprint(N)
    log('region %s, branch nodes %d, tree %s' % (R, len(todo), fp))
    t0 = time.time()
    try:
        errf = open(out + '.err', 'wb')   # KataGo's own log
    except OSError as e:
        log('ERROR cannot write next to %s: %s' % (out, e))
        return 5
    try:
        eng = Engine(engine_cmd(c), errf, c['stall'])
    except OSError as e:
        errf.close()
        log('ERROR cannot start KataGo: %s (set WEIQI_KATAGO_DIR / WEIQI_KATAGO_MODEL)' % e)
        return 5
    prog = Progress(out)
    part, done, state = out + '.part', {}, {'report_as': None}

    def save():
        if done:
            atomic_write_json(part, {'tree_fp': fp, 'visits': [VB, VF], 'report_as': state['report_as'], 'done': done})

    try:
        report_as, source = detect_report_as(eng, c['config'], log)
        state['report_as'] = report_as
        log('score perspective of the engine: %s (%s), stored as side to move' % (report_as, source))
        done.update(load_part(part, fp, [VB, VF], report_as))
        if done:
            log('resuming: %d results from an earlier run' % len(done))
        conv = lambda d: convert(d, report_as)
        base_of = lambda q: q.split('_')[0]
        # phase 1: main search at every branch point (+ a control run without the stones outside the region)
        reqs = []
        for i in todo:
            reqs.append(query(N, R, allow, i, True, '%d' % i, VB))
            if any(not inreg(R, *k) for k in stones_at(N, i)):
                reqs.append(query(N, R, allow, i, False, '%d_0' % i, min(VB, 200)))
        already = sum(1 for r in reqs if r['id'] in done)
        prog.start('phase 1/2 main search', len(reqs), already, len(todo), len({base_of(r['id']) for r in reqs if r['id'] in done}))
        run_queries(eng, reqs, done, conv, save, prog, c['window'], base_of)
        res = {r['id']: done[r['id']] for r in reqs}
        # phase 2: the teacher's move at each branch point, if the main search did not look at it enough
        reqs2, need = [], []
        for i in todo:
            a = res.get('%d' % i)
            if not a or 'cand' not in a:
                continue
            tv = max(m['v'] for m in a['cand'])
            for ch in live_children(N, i):
                if N[ch]['mv'] is None:
                    continue
                m = gtp(N[ch]['mv'][0] % C, N[ch]['mv'][0] // C)
                cm = next((x for x in a['cand'] if x['m'] == m), None)
                if cm and cm['v'] >= max(10, .1 * tv):
                    continue
                if m not in allow:
                    continue  # outside the region (tenuki): not evaluated
                reqs2.append(query(N, R, allow, ch, True, 'f%d' % ch, VF))
                need.append((i, ch, m))
        prog.start('phase 2/2 teacher moves', len(reqs2), sum(1 for r in reqs2 if r['id'] in done), len(reqs2), 0)
        run_queries(eng, reqs2, done, conv, save, prog, c['window'], lambda q: q)
        for i, ch, m in need:
            f, a = done.get('f%d' % ch), res['%d' % i]
            if not f or 'cand' not in f or not f['cand']:
                continue
            t = f['cand'][0]
            a['cand'].append({'m': m, 'v': VF, 's': round(-f['score'], 1), 'wr': round(1 - f['wr'], 3), 'p': -1, 'pv': [m] + t['pv'][:7], 'f': 1})
    except EngineError as e:
        save()
        log('ERROR %s. %d results are kept in %s, run again to continue.' % (e, len(done), part))
        eng.close()
        errf.close()
        return 4
    except BaseException:
        save()
        eng.close()
        errf.close()
        raise
    eng.close()
    errf.close()
    errors = sum(1 for v in res.values() if 'err' in v)
    res['_meta'] = make_meta(N, visits=[VB, VF], region=R, complete=not limit, report_as=report_as, report_as_source=source,
                             errors=errors, queries=len(res), elapsed_s=round(time.time() - t0))
    atomic_write_json(out, res)
    try:
        os.remove(part)
    except OSError:
        pass
    log('done in %ds, %d entries, %d errors' % (time.time() - t0, len(res) - 1, errors))
    sus, stats = perspective_suspect(res)
    if sus:
        log('ERROR the scores look like Black perspective (first candidate is not the best one: %s). They will not be merged.' % stats)
        return 3
    return 0


def main(argv=None):
    setup_stdio()
    argv = argv if argv is not None else sys.argv
    if len(argv) < 3:
        print(__doc__, file=sys.stderr)
        return 1
    N = read_json(argv[1])
    VB = int(argv[3]) if len(argv) > 3 else 220
    VF = int(argv[4]) if len(argv) > 4 else 100
    return analyze(N, argv[2], VB, VF, limit=int(os.environ.get('LIMIT', '0')))


if __name__ == '__main__':
    sys.exit(main())

In [ ]:
%%writefile analyze_remote.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Cloud batch runner for the KataGo analysis of variation trees (Colab, Kaggle, Slurm nodes).

It does NOT carry a copy of the analysis logic any more. Every group is analysed by pipeline/analyze2.py, the same
module the local pipeline runs, so the result is the same kind of file by construction:
  * scores and winrates stored from the side to move (reportAnalysisWinratesAs=BLACK is converted in analyze2.convert),
    with a live perspective check against the engine;
  * region, branch points, 220 main visits, 100 forced visits for the teacher's moves, control search without outside stones;
  * a window of queries in flight (never everything at once, so the pipe cannot deadlock), a checkpoint (<out>.part) so a
    killed run resumes, a stall timeout so a silent engine ends the run with a message;
  * a "_meta" block with the tree fingerprint, schema, perspective and the complete flag, so wq.py check, treesync.check_analysis
    and orch2 accept the file as it is. No wq.py stamp is needed.
analyze2.py, treesync.py and wqutil.py are looked up next to this file (the notebook writes them there) or in ../pipeline.

Input : <in>/g_<name>.n.json (the tree the local pipeline wrote), optionally <in>/manifest.json from pack_inputs.py
        (processing order and the tree fingerprints, which are checked before anything is run)
Output: <out>/g_<name>.a2.json            the analysis, exactly what analyze2.py writes
        <out>/g_<name>.a2.meta.json       side note for humans: seconds, queries, seconds per query, KataGo version, weights hash
        <out>/run_summary.json            one entry per group of this run

Usage:
  python analyze_remote.py --in ./in --out ./out --katago ./katago --model ./model.bin.gz [--expect-sha256 HASH]
A group that already has a valid a2.json is skipped, a killed run continues from its checkpoint, --force recomputes.
Exit code: 0 all groups fine, 1 a group failed or needs a look, 2 bad input, 4 engine failed (re-run to continue), 5 engine missing.
"""
import argparse
import glob
import hashlib
import json
import os
import re
import shlex
import subprocess
import sys
import tempfile
import time

HERE = os.path.dirname(os.path.abspath(__file__))


def _pipeline_dir():
    """Where analyze2.py, treesync.py and wqutil.py live: next to this file (notebook) or in the repo's pipeline/ folder."""
    for d in (HERE, os.path.join(os.path.dirname(HERE), 'pipeline')):
        if all(os.path.exists(os.path.join(d, f)) for f in ('analyze2.py', 'treesync.py', 'wqutil.py')):
            return d
    sys.exit('analyze2.py, treesync.py and wqutil.py not found next to %s or in ../pipeline' % __file__)


sys.path.insert(0, _pipeline_dir())
import analyze2 as an  # noqa: E402
import treesync as ts  # noqa: E402
from wqutil import atomic_write_json, read_json, setup_stdio  # noqa: E402

TREE_KEYS = ('id', 'par', 'mv', 'ch', 'dead', 'setup', 'pos')

# Same parameters as the local analysis.cfg (checked against it); only the log directory differs.
CFG = """# Written by analyze_remote.py; the search parameters match the local analysis.cfg, only the log directory differs
logDir = {logdir}
reportAnalysisWinratesAs = BLACK
maxVisits = 500
numAnalysisThreads = {nat}
numSearchThreadsPerAnalysisThread = {nst}
nnMaxBatchSize = {bs}
nnCacheSizePowerOfTwo = 23
nnMutexPoolSizePowerOfTwo = 17
nnRandomize = true
"""


# ---------------------------------------------------------------- input checks

def validate_tree(N):
    """Raise ValueError with a plain message when N is not a tree analyze2 can use (fingerprint and indexing need these fields)."""
    if not isinstance(N, list) or not N or not all(isinstance(n, dict) for n in N):
        raise ValueError('not a list of nodes')
    for k, n in enumerate(N):
        missing = [f for f in TREE_KEYS if f not in n]
        if missing:
            raise ValueError('node %d lacks %s' % (k, ', '.join(missing)))
        if n['id'] != k:
            raise ValueError('node at position %d has id %r (analyze2 indexes the list by id)' % (k, n['id']))


def sha256_of(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def engine_version(katago):
    """First lines of `katago version` (prints text only, does not load a network), or ''."""
    try:
        r = subprocess.run([katago, 'version'], capture_output=True, text=True, encoding='utf-8', errors='replace', timeout=60)
        return ' | '.join(x.strip() for x in r.stdout.splitlines()[:4] if x.strip())
    except (OSError, subprocess.SubprocessError):
        return ''


def widen_startup_wait(seconds):
    """analyze2 waits 120 s for the answer to its perspective self test, and that wait includes loading the network.

    A cold cloud GPU can need longer (CUDA kernels are compiled at start), so the wait is raised here instead of editing
    pipeline/analyze2.py, which must stay identical to the local copy. Calling it again replaces the earlier value."""
    orig = getattr(an.Engine, '_wq_orig_get_for', None) or an.Engine.get_for

    def get_for(self, qid, timeout):
        return orig(self, qid, seconds if qid == 'selftest' else timeout)

    an.Engine._wq_orig_get_for = orig
    an.Engine.get_for = get_for


def side_to_move(N, i, p):
    """'B' or 'W' for node i: the opposite of the last move on its path, else the colour the teacher plays next
    (majority of the live children, a tie goes to B), else 'B'. Same rule as the fix in the local analyze2.py."""
    if p:
        return 'W' if N[p[-1]]['mv'][1] == 'B' else 'B'
    cols = [N[c]['mv'][1] for c in an.live_children(N, i) if N[c]['mv']]
    return max(sorted(set(cols)), key=cols.count) if cols else 'B'


def fix_initial_player():
    """Bug fix: a request without moves carried no initialPlayer, so KataGo assumed Black and a position set up from stones
    where the teacher continues with White was analysed for the wrong side (the scores were then converted for Black too).

    The request is completed here, for requests without moves only, and only when analyze2 has not set the player itself,
    so this stays a no-op once pipeline/analyze2.query() does it. Requests that have moves are not touched. Calling it
    again does not stack the wrapper."""
    orig = getattr(an, '_wq_orig_query', None) or an.query

    def query(N, R, allow, i, ctx, qid, visits):
        req = orig(N, R, allow, i, ctx, qid, visits)
        if not req['moves'] and 'initialPlayer' not in req:
            req['initialPlayer'] = side_to_move(N, i, an.path_of(N, i)[1])
        return req

    an._wq_orig_query = orig
    an.query = query


def order_files(files, manifest):
    """Manifest order first (pack_inputs.py puts the groups the local run reaches last at the front), the rest by name."""
    names = (manifest or {}).get('order') or []
    rank = {'g_%s' % n: i for i, n in enumerate(names)}
    return sorted(files, key=lambda f: (rank.get(os.path.basename(f)[:-len('.n.json')], len(rank)), os.path.basename(f)))


# ---------------------------------------------------------------- one group

def entry_stats(A):
    """(branch nodes, forced evaluations, queries) of a finished analysis, for the side file."""
    meta = A.get('_meta') or {}
    forced = sum(1 for k, a in A.items() if k != '_meta' and isinstance(a, dict) for c in a.get('cand', []) if c.get('f'))
    nodes = sum(1 for k in A if re.fullmatch(r'\d+', k))
    return nodes, forced, meta.get('queries', 0)


def run_group(path, outp, a, info):
    """Analyse one tree. Returns (status, exit code to stop with or 0). Status: done, skipped, failed, suspect."""
    name = os.path.basename(path)[:-len('.n.json')]
    try:
        N = read_json(path)
        validate_tree(N)
    except (ValueError, OSError) as e:
        print('  ERROR %s: %s' % (path, e), flush=True)
        return 'failed', 0
    fp = ts.tree_fingerprint(N)
    want = (info.get('groups') or {}).get(name[2:], {}).get('tree_fp')
    if want and want != fp:
        print('  ERROR %s changed after packing (fingerprint %s, manifest says %s): pack it again' % (name, fp, want), flush=True)
        return 'failed', 0
    old = read_json(outp, default=None) if os.path.exists(outp) else None
    if old is not None and not a.force:
        st, msg = ts.check_analysis(N, old)
        if st == 'ok':
            print('  skip (exists, valid): %s' % outp, flush=True)
            return 'skipped', 0
        print('  existing %s is %s (%s): moved aside, running again' % (os.path.basename(outp), st, msg), flush=True)
        os.replace(outp, outp + '.stale')
    elif old is None and os.path.exists(outp):
        os.replace(outp, outp + '.stale')   # unreadable (half written by something else): keep it for inspection, never reuse
    if a.force and os.path.exists(outp + '.part'):
        os.remove(outp + '.part')    # --force means from scratch, not from an old checkpoint
    t0 = time.time()
    rc = an.analyze(N, outp, a.visits, a.forced, limit=a.limit)
    dt = time.time() - t0
    if rc in (4, 5):
        return 'failed', rc          # the engine itself is the problem: stop the batch, a re-run continues from the checkpoint
    if rc == 2:
        return 'failed', 0
    if rc == 3:                      # perspective looks wrong: analyze2 wrote the file but it must never travel back
        os.replace(outp, outp + '.suspect')
        return 'suspect', 0
    A = read_json(outp)
    st, msg = ts.check_analysis(N, A)
    if st != ('incomplete' if a.limit else 'ok'):
        print('  ERROR the finished file is not accepted by treesync.check_analysis: %s %s' % (st, msg), flush=True)
        os.replace(outp, outp + '.rejected')
        return 'failed', 0
    nodes, forced, queries = entry_stats(A)
    total = queries + forced
    meta = {'group': name, 'seconds': round(dt), 'queries_main': queries, 'queries_forced': forced, 'queries_total': total,
            'sec_per_query': round(dt / total, 2) if total else None, 'branch_nodes': nodes, 'region': A['_meta'].get('region'),
            'visits': a.visits, 'forced_visits': a.forced, 'errors': A['_meta'].get('errors', 0), 'report_as': A['_meta'].get('report_as'),
            'tree_fp': fp, 'finished': time.strftime('%Y-%m-%d %H:%M:%S'),
            'note': 'seconds covers the last run only; a resumed group counts from the resume'}
    meta.update(info.get('run', {}))
    atomic_write_json(outp[:-len('.json')] + '.meta.json', meta, ensure_ascii=False, indent=1)
    print('  done %s: %ds, %d main + %d forced queries, %.1f s per query' % (name, dt, queries, forced, dt / total if total else 0), flush=True)
    if meta['errors']:
        print('  WARNING %d queries returned an error: those nodes have no AI data' % meta['errors'], flush=True)
        return 'failed', 0
    return 'done', 0


# ---------------------------------------------------------------- command line

def build_engine(a, outdir):
    """Put the engine command and the settings analyze2 reads from the environment in place. Returns (info about the engine, exit code)."""
    run = {}
    if a.engine_cmd:
        os.environ['WEIQI_KATAGO_CMD'] = json.dumps(shlex.split(a.engine_cmd))
        run['engine'] = 'custom command (tests)'
    else:
        if not a.model or not os.path.exists(a.model):
            print('Model file not found: %r (pass the .bin.gz with --model)' % a.model, file=sys.stderr)
            return None, 2
        cfgdir = os.path.join(outdir, 'katago_cfg')
        os.makedirs(cfgdir, exist_ok=True)
        cfg = os.path.join(cfgdir, 'analysis.cfg')   # the file name analyze2 expects inside WEIQI_KATAGO_DIR
        if a.cfg:
            txt = open(a.cfg, encoding='utf-8', errors='replace').read()
            if not re.search(r'^\s*reportAnalysisWinratesAs\s*=\s*BLACK\b', txt, re.M):
                print('warning: the supplied cfg has no reportAnalysisWinratesAs = BLACK; it is forced with -override-config.', file=sys.stderr)
            open(cfg, 'w', encoding='utf-8').write(txt)
        else:
            # KataGo's own log goes to the local temp folder: the output folder may be a synced Drive folder
            logdir = os.path.join(tempfile.gettempdir(), 'katago_logs').replace('\\', '/')
            open(cfg, 'w', encoding='utf-8').write(CFG.format(logdir=logdir, nat=a.threads, nst=a.search_threads, bs=a.batch))
        os.environ['WEIQI_KATAGO_DIR'] = cfgdir
        os.environ['WEIQI_KATAGO_MODEL'] = a.model
        os.environ['WEIQI_KATAGO_CMD'] = json.dumps([a.katago, 'analysis', '-model', a.model, '-config', cfg,
                                                     '-override-config', 'reportAnalysisWinratesAs=BLACK'])
        run['katago'] = engine_version(a.katago)
        run['model'] = os.path.basename(a.model)
        run['model_bytes'] = os.path.getsize(a.model)
        run['model_sha256'] = sha256_of(a.model)
        print('KataGo:', run['katago'] or '(version unknown)')
        print('weights: %s, %d bytes, sha256 %s' % (run['model'], run['model_bytes'], run['model_sha256']), flush=True)
        if a.expect_sha256 and a.expect_sha256.lower() != run['model_sha256']:
            print('The weights file is not the one the local pipeline uses (expected sha256 %s). Results would not be comparable '
                  'with the existing analysis. Upload the right file, or drop --expect-sha256 if you really mean it.' % a.expect_sha256, file=sys.stderr)
            return None, 2
    os.environ['WEIQI_WINDOW'] = str(a.window)
    os.environ['WEIQI_STALL_S'] = str(a.stall)
    widen_startup_wait(a.start_timeout)
    fix_initial_player()
    run['initial_player'] = 'explicit for requests without moves (side_to_move)'
    return {'run': run}, 0


def main(argv=None):
    setup_stdio()
    ap = argparse.ArgumentParser(description='Cloud batch runner around pipeline/analyze2.py')
    ap.add_argument('--in', dest='inp', required=True, help='directory containing g_*.n.json, or a single n.json file')
    ap.add_argument('--out', required=True, help='output directory (g_*.a2.json is written here)')
    ap.add_argument('--katago', default=os.environ.get('KATAGO_BIN', './katago'))
    ap.add_argument('--model', default=os.environ.get('KATAGO_MODEL', ''))
    ap.add_argument('--expect-sha256', default='', help='refuse to run when the weights file has another sha256 (same network as the local machine)')
    ap.add_argument('--cfg', default='', help='your own analysis.cfg; generated automatically if omitted (reportAnalysisWinratesAs=BLACK is forced anyway)')
    ap.add_argument('--engine-cmd', default='', help='(for tests) use this command instead of katago, e.g. "python fake_katago.py"')
    ap.add_argument('--visits', type=int, default=220, help='main search visits (local baseline 220)')
    ap.add_argument('--forced', type=int, default=100, help='forced evaluation visits (local baseline 100)')
    ap.add_argument('--threads', type=int, default=2, help='numAnalysisThreads (can be raised to 3~4 on a strong GPU)')
    ap.add_argument('--search-threads', type=int, default=16)
    ap.add_argument('--batch', type=int, default=64, help='nnMaxBatchSize')
    ap.add_argument('--window', type=int, default=8, help='queries in flight at once (never write everything up front)')
    ap.add_argument('--stall', type=float, default=900, help='seconds without any answer before the run gives up')
    ap.add_argument('--start-timeout', type=float, default=600, help='seconds to wait for the first answer (network load plus self test)')
    ap.add_argument('--limit', type=int, default=int(os.environ.get('LIMIT', '0')), help='analyze only the first N branch points per group (trial run, the result is marked incomplete)')
    ap.add_argument('--force', action='store_true', help='recompute even if a valid a2.json already exists')
    ap.add_argument('--only', default='', help='only process groups whose file name contains this substring')
    a = ap.parse_args(argv)

    files = [a.inp] if os.path.isfile(a.inp) else sorted(glob.glob(os.path.join(a.inp, 'g_*.n.json')))
    manifest = read_json(os.path.join(a.inp, 'manifest.json'), default=None) if os.path.isdir(a.inp) else None
    if a.only:
        files = [f for f in files if a.only in os.path.basename(f)]
    if not files:
        print('No g_*.n.json found in %s' % a.inp, file=sys.stderr)
        return 2
    files = order_files(files, manifest)
    os.makedirs(a.out, exist_ok=True)
    info, rc = build_engine(a, a.out)
    if rc:
        return rc
    info['groups'] = (manifest or {}).get('groups', {})
    print('%d groups to look at, window %d, %d/%d visits' % (len(files), a.window, a.visits, a.forced), flush=True)
    summary, T0, stop = [], time.time(), 0
    for k, f in enumerate(files, 1):
        name = os.path.basename(f)[:-len('.n.json')]
        print('[%d/%d] %s' % (k, len(files), name), flush=True)
        t0 = time.time()
        status, stop = run_group(f, os.path.join(a.out, name + '.a2.json'), a, info)
        summary.append({'group': name, 'status': status, 'seconds': round(time.time() - t0)})
        if stop:
            print('Stopping: the engine failed. Everything answered so far is kept in the .part file, run the same command again to continue.', flush=True)
            break
    atomic_write_json(os.path.join(a.out, 'run_summary.json'), {'finished': time.strftime('%Y-%m-%d %H:%M:%S'), 'total_seconds': round(time.time() - T0),
                                                                'groups': summary}, ensure_ascii=False, indent=1)
    bad = [s['group'] for s in summary if s['status'] in ('failed', 'suspect')]
    print('summary: %s' % ', '.join('%s=%d' % (st, sum(1 for s in summary if s['status'] == st)) for st in ('done', 'skipped', 'failed', 'suspect')))
    if bad:
        print('groups that need a look: %s' % ', '.join(bad))
    print('all done ->', os.path.abspath(a.out))
    return stop or (1 if bad else 0)


if __name__ == '__main__':
    sys.exit(main())

In [ ]:
%%writefile setup_katago.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Download and check the Linux GPU build of KataGo (works on Colab / Kaggle / Slurm nodes). No root needed, no system libraries are touched.

What it does:
  1) Downloads katago-<ver>-<build>-linux-x64.zip from the GitHub release and unpacks it (default v1.18.2 + cuda12.1-cudnn9.8.0:
     it is built against the lowest CUDA version, so it is the most forward-compatible with the newer CUDA 12.x runtimes in
     cloud images; since v1.18.x, Transformer networks need cuDNN >= 9.8);
  2) uses ldd to find missing shared libraries; if cuDNN/cuBLAS/cudart/nvrtc are missing, installs them with pip into a
     separate directory (--target, system Python packages are untouched) and builds the matching LD_LIBRARY_PATH;
  3) locates the weights file (given with --model, or searched for in the usual places; otherwise --weights-url is required);
  4) writes <dir>/env.sh and <dir>/env.json, and finally prints KATAGO_BIN / KATAGO_MODEL / LD_LIBRARY_PATH.

References (checked 2026-11): KataGo v1.18.2 release page https://github.com/lightvector/KataGo/releases/tag/v1.18.2
  - "Use CUDNN >= 9.8.0. The CUDNN 8.9.7 builds will be a LOT slower when running transformers"
  - "TensorRT versions older than 10 are not supported."
  - v1.18.2 adds tensor-core attention for Turing (RTX 20xx / Tesla T4, sm_75), speeding up Transformer networks by more than 50%.
  5) checks the downloaded zip against the sha256 published on the release page (table KNOWN_SHA256 below).
Note: this has not been run on real hardware (the environment it was written in had no GPU); the version combination was picked
from the official release notes. The asset names and checksums were read from the release data on 2026-11-08; nothing was downloaded. If something fails, see the "Troubleshooting" section of README.md.
"""
import argparse, glob, hashlib, json, os, shutil, subprocess, sys, urllib.request, zipfile

# sha256 of the official Linux release zips of KataGo v1.18.2, copied from the GitHub release data on 2026-11-08.
# A download that does not match is deleted and the run stops: a broken or swapped binary must not be executed.
# Other versions or builds are not in this table and are only reported as "not verified".
KNOWN_SHA256 = {
    'katago-v1.18.2-cuda12.1-cudnn9.8.0-linux-x64.zip': '3e30b486b7bc38287eeead350aa832ff8ca50d1f0c0a83b5409cabf8a3bc9c5d',
    'katago-v1.18.2-cuda12.5-cudnn9.8.0-linux-x64.zip': '8c39d9ac1053f0e0eff0b0d376fcc2b7503e8caf44020710dda758de53a802f8',
    'katago-v1.18.2-cuda12.8-cudnn9.8.0-linux-x64.zip': '2cb197fc9bc2050f29112abb5b83cdbad63936cb4605d6e199772aedc9806c02',
}


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def verify_zip(path, name, enforce=True):
    """True when the zip matches the published checksum or none is known; on a mismatch the file is removed and False returned."""
    want = KNOWN_SHA256.get(name)
    if not want:
        print('No published checksum known for %s: not verified.' % name)
        return True
    got = sha256_file(path)
    if got == want:
        print('Checksum of %s matches the release (sha256 %s...)' % (name, got[:16]))
        return True
    print('!! Checksum mismatch for %s' % name)
    print('   expected %s' % want)
    print('   got      %s' % got)
    if enforce:
        os.remove(path)
        print('   The file was deleted. Try again; if it keeps failing, use another --build or pass --no-verify only if you know why.')
        return False
    return True


def sh(cmd, env=None):
    return subprocess.run(cmd, capture_output=True, text=True, env=env)


def download(url, dst):
    print('Downloading', url, flush=True)
    req = urllib.request.Request(url, headers={'User-Agent': 'josekiapp-remote/1'})
    with urllib.request.urlopen(req) as r, open(dst + '.part', 'wb') as f:
        total = int(r.headers.get('Content-Length', '0') or 0)
        got = 0
        while True:
            b = r.read(1 << 20)
            if not b:
                break
            f.write(b)
            got += len(b)
            if total:
                print('\r  %d / %d MB' % (got >> 20, total >> 20), end='', flush=True)
    print()
    os.replace(dst + '.part', dst)


def missing_libs(binary, env):
    r = sh(['ldd', binary], env)
    return sorted({ln.split('=>')[0].strip() for ln in r.stdout.splitlines() if 'not found' in ln}), r.stdout


def find_model(explicit):
    if explicit:
        return explicit if os.path.exists(explicit) else ''
    pats = ['./kata1-tf3-b11c768*.bin.gz', './*.bin.gz', '/content/*.bin.gz', '/content/drive/MyDrive/josekiapp/*.bin.gz',
            '/content/drive/MyDrive/*.bin.gz', '/kaggle/input/*/*.bin.gz', '/kaggle/input/*/*/*.bin.gz', os.path.expanduser('~/*.bin.gz')]
    for p in pats:
        hit = sorted(glob.glob(p))
        if hit:
            return hit[0]
    return ''


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--dir', default='katago_env')
    ap.add_argument('--ver', default=os.environ.get('KATAGO_VER', 'v1.18.2'))
    ap.add_argument('--build', default=os.environ.get('KATAGO_BUILD', 'cuda12.1-cudnn9.8.0'),
                    help='e.g. cuda12.1-cudnn9.8.0 / cuda12.5-cudnn9.8.0 / cuda12.8-cudnn9.8.0 / cuda13.2-cudnn9.24.0')
    ap.add_argument('--model', default=os.environ.get('KATAGO_MODEL', ''))
    ap.add_argument('--weights-url', default=os.environ.get('WEIGHTS_URL', ''))
    ap.add_argument('--no-verify', action='store_true', help='do not stop on a checksum mismatch of the KataGo zip (it is still reported)')
    ap.add_argument('--no-pip-libs', action='store_true', help='do not try to pip-install the NVIDIA runtime libraries when libraries are missing')
    a = ap.parse_args()

    d = os.path.abspath(a.dir)
    os.makedirs(d, exist_ok=True)
    nv = sh(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'])
    print('GPU:', nv.stdout.strip() or '(nvidia-smi unavailable -- no GPU runtime? Colab: Runtime > Change runtime type > GPU)')

    binary = os.path.join(d, 'katago')
    if not os.path.exists(binary):
        name = 'katago-%s-%s-linux-x64.zip' % (a.ver, a.build)
        z = os.path.join(d, name)
        download('https://github.com/lightvector/KataGo/releases/download/%s/%s' % (a.ver, name), z)
        if not verify_zip(z, name, enforce=not a.no_verify):
            sys.exit(1)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(d)
        os.chmod(binary, 0o755)
    print('KataGo:', binary)

    env = dict(os.environ)
    miss, out = missing_libs(binary, env)
    libpaths = []
    if miss and not a.no_pip_libs:
        print('Missing shared libraries:', miss)
        nvdir = os.path.join(d, 'nvlibs')
        pk = ['nvidia-cudnn-cu12', 'nvidia-cublas-cu12', 'nvidia-cuda-runtime-cu12', 'nvidia-cuda-nvrtc-cu12']
        print('Installing with pip into a separate directory %s (system packages are not affected): %s' % (nvdir, ' '.join(pk)), flush=True)
        r = sh([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--target', nvdir] + pk)
        if r.returncode:
            print(r.stderr[-800:])
        libpaths = sorted(glob.glob(os.path.join(nvdir, 'nvidia', '*', 'lib')))
        env['LD_LIBRARY_PATH'] = ':'.join(libpaths + [env.get('LD_LIBRARY_PATH', '')]).strip(':')
        miss, out = missing_libs(binary, env)
    if miss:
        # system libraries such as libzip: try apt when running as root (works on Colab, not on clusters)
        if hasattr(os, 'geteuid') and os.geteuid() == 0 and any('zip' in m for m in miss) and shutil.which('apt-get'):
            for pkg in ('libzip4', 'libzip4t64', 'libzip5'):
                if sh(['apt-get', 'install', '-y', '-qq', pkg]).returncode == 0:
                    break
            miss, out = missing_libs(binary, env)
    if miss:
        print('Still missing:', miss, '\n--- ldd ---\n' + out)
        print('=> See "Troubleshooting" in README.md: try another --build (cuda12.5/12.8), use the OpenCL/Eigen build, or ask the cluster admin to load the cuda/cudnn modules.')
    else:
        print('ldd check passed (no missing shared libraries)')

    model = find_model(a.model)
    if not model and a.weights_url:
        model = os.path.join(d, os.path.basename(a.weights_url.split('?')[0]))
        if not os.path.exists(model):
            download(a.weights_url, model)
    if not model:
        print('!! No weights file (*.bin.gz) found. Upload the same kata1-tf3-b11c768-s12752M-d6531M.bin.gz as on the local machine (about 262MB), or set WEIGHTS_URL / --weights-url.')
    else:
        print('Weights:', model, '(%d MB)' % (os.path.getsize(model) >> 20))

    ld = env.get('LD_LIBRARY_PATH', '')
    info = {'katago': binary, 'model': model, 'ld_library_path': ld, 'missing_libs': miss, 'ver': a.ver, 'build': a.build}
    json.dump(info, open(os.path.join(d, 'env.json'), 'w'), indent=1)
    open(os.path.join(d, 'env.sh'), 'w').write('export LD_LIBRARY_PATH="%s"\nexport KATAGO_BIN="%s"\nexport KATAGO_MODEL="%s"\n' % (ld, binary, model))
    print('Wrote %s/env.sh (bash: source it)' % d)
    print(json.dumps(info, ensure_ascii=False))


if __name__ == '__main__':
    main()

## 2. 输入：变化树和权重 / Input: the trees and the weights

笔记本会在这些位置找 `n_json_*.zip`（或散放的 `g_*.n.json`）和权重文件：工作目录、Colab 的 `MyDrive/josekiapp/`、Kaggle 的 `/kaggle/input/*/`。找不到时，Colab 会弹出上传框。权重文件会用 sha256 核对，必须与本机用的是同一个文件。

The notebook looks for `n_json_*.zip` (or loose `g_*.n.json` files) and the weights in these places: the working directory, `MyDrive/josekiapp/` on Colab, and `/kaggle/input/*/` on Kaggle. If it finds nothing, Colab shows an upload box. The weights are checked by sha256 and must be the very file the local machine uses.

In [ ]:
DRIVE_BASE = ''
if ON_COLAB and USE_DRIVE:
    from google.colab import drive   # a Google sign-in and permission prompt appears; you approve it yourself
    drive.mount('/content/drive')
    DRIVE_BASE = '/content/drive/MyDrive/' + DRIVE_DIR
    os.makedirs(DRIVE_BASE, exist_ok=True)
IN_DIR = os.path.join(WORK, 'in')
OUT_DIR = os.path.join(DRIVE_BASE, 'out') if DRIVE_BASE else os.path.join(WORK, 'out')
for d in (IN_DIR, OUT_DIR, WORK + '/uploads', WORK + '/weights'):
    os.makedirs(d, exist_ok=True)


def search_roots():
    '''(folder, recursive) pairs. /content itself is not searched recursively: Drive is mounted below it.'''
    r = [(WORK, False), (WORK + '/uploads', False)]
    if DRIVE_BASE:
        r.append((DRIVE_BASE, True))
    r += [(d, True) for d in sorted(glob.glob('/kaggle/input/*'))]
    return r


def find_files(pattern):
    hits = []
    for root, rec in search_roots():
        hits += glob.glob(os.path.join(root, '**', pattern) if rec else os.path.join(root, pattern), recursive=rec)
    return sorted(set(hits))


def unpack_zips():
    '''Take the trees and manifest.json out of every zip that has them (several packages are merged, in zip name order), and a
    weights file out of a zip that has one. Basename only: no path tricks.'''
    man, found = {'order': [], 'groups': {}}, 0
    for zp in find_files('*.zip'):
        try:
            z = zipfile.ZipFile(zp)
        except zipfile.BadZipFile:
            continue
        with z:
            for member in z.namelist():
                base = os.path.basename(member)
                if base.endswith('.n.json') and base.startswith('g_'):
                    open(os.path.join(IN_DIR, base), 'wb').write(z.read(member))
                elif base == 'manifest.json' and 'n_json' in os.path.basename(zp):
                    part = json.loads(z.read(member).decode('utf-8-sig'))
                    found += 1
                    man['order'] += [n for n in part.get('order', []) if n not in man['order']]
                    man['groups'].update(part.get('groups', {}))
                elif base.endswith('.bin.gz') and not glob.glob(WORK + '/weights/*.bin.gz'):
                    print('extracting the weights from', zp)
                    with z.open(member) as src, open(os.path.join(WORK, 'weights', base), 'wb') as dst:
                        shutil.copyfileobj(src, dst)
    if found:
        open(os.path.join(IN_DIR, 'manifest.json'), 'w', encoding='utf-8').write(json.dumps(man, ensure_ascii=False))
    if found > 1:
        print('%d packages with a manifest.json were merged (the groups of the first package come first)' % found)


def collect_inputs():
    for f in find_files('g_*.n.json'):
        if os.path.dirname(f) != IN_DIR:
            shutil.copy(f, IN_DIR)
    unpack_zips()
    return sorted(glob.glob(IN_DIR + '/g_*.n.json'))


inp = collect_inputs()
if not inp and ON_COLAB:
    from google.colab import files
    print('Nothing found. Select n_json_<date>.zip (or the g_*.n.json files) to upload:')
    for name, data in files.upload().items():
        open(os.path.join(WORK, 'uploads', name), 'wb').write(data)
    inp = collect_inputs()
man = {}
if os.path.exists(IN_DIR + '/manifest.json'):
    man = json.load(open(IN_DIR + '/manifest.json', encoding='utf-8-sig'))
print('%d input groups (the order in which they will be analysed):' % len(inp))
for n in man.get('order') or [os.path.basename(f)[2:-7] for f in inp]:
    g = (man.get('groups') or {}).get(n, {})
    print('  %-40s nodes %s, branch points %s, main queries %s' % (n, g.get('nodes', '?'), g.get('branch_nodes', '?'), g.get('queries_main', '?')))
assert inp, 'No g_*.n.json found. Put n_json_<date>.zip into MyDrive/%s (Colab) or add it as a Kaggle Dataset, then run this cell again.' % DRIVE_DIR


def sha256_of(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def find_weights():
    cands = [WEIGHTS_PATH] if WEIGHTS_PATH else []
    cands += find_files('*.bin.gz')
    return [c for c in dict.fromkeys(cands) if os.path.isfile(c)]


unpack_zips()
cands = find_weights()
if not cands and ON_COLAB:
    from google.colab import files
    print('Weights not found. Select %s (about 262 MB, the upload is slow):' % WEIGHTS_NAME)
    for name, data in files.upload().items():
        open(os.path.join(WORK, 'uploads', name), 'wb').write(data)
    cands = find_weights()
assert cands, 'No weights file (*.bin.gz) found. Put %s into MyDrive/%s (Colab) or add it as a Kaggle Dataset.' % (WEIGHTS_NAME, DRIVE_DIR)
MODEL = ''
for c in cands:
    got = sha256_of(c)
    ok = (not EXPECTED_SHA256) or got == EXPECTED_SHA256
    print('weights candidate: %s  %d bytes  sha256 %s  %s' % (c, os.path.getsize(c), got, 'MATCHES the local file' if ok else 'is NOT the local file'))
    if ok and not MODEL:
        MODEL = c
assert MODEL, 'None of the weight files has sha256 %s. Upload exactly the file the local pipeline uses (%s, %d bytes).' % (EXPECTED_SHA256, WEIGHTS_NAME, EXPECTED_BYTES)
if MODEL.startswith('/content/drive'):   # read it from local disk: faster and does not depend on the Drive connection
    local = os.path.join(WORK, 'weights', os.path.basename(MODEL))
    if not os.path.exists(local) or os.path.getsize(local) != os.path.getsize(MODEL):
        shutil.copy(MODEL, local)
    MODEL = local
print('using the weights', MODEL)

## 3. 安装 KataGo（GPU 版） / Install KataGo (GPU build)

脚本会从 GitHub 官方发布页下载 KataGo v1.18.2 的 Linux CUDA 版，并用发布页公布的 sha256 校验；缺少的 CUDA 库用 pip 装到单独的目录（不动系统自带的包）。

The script downloads the official Linux CUDA build of KataGo v1.18.2 from the GitHub release page and checks it against the sha256 published there; missing CUDA libraries are pip-installed into a separate directory (the system packages are not touched).

In [ ]:
cmd = [sys.executable, '-u', 'setup_katago.py', '--dir', WORK + '/katago_env', '--build', KATAGO_BUILD, '--model', MODEL]
assert run_stream(cmd) == 0, 'setup_katago.py failed: see the messages above (README_cloud.md, troubleshooting)'
ENV = json.load(open(WORK + '/katago_env/env.json'))
assert not ENV['missing_libs'], 'missing shared libraries: %s (try another KATAGO_BUILD in the settings cell)' % ENV['missing_libs']
os.environ['LD_LIBRARY_PATH'] = ENV['ld_library_path']
print('Ready:', ENV['katago'])
print(sh('%s version' % ENV['katago']))

## 4. （可选，建议）试跑 3 个分叉点，看速度 / (Optional, recommended) Trial run on 3 branch points to see the speed

本机核显每个主搜索（220 visits）约 14 秒，老师着法的强制评估（100 visits）约 4.5 秒（来自本机的进度日志）。试跑同时检查整条链路：KataGo 能启动、视角自检结果是 `BLACK (selftest)`、输出文件通过 `treesync.check_analysis`。第一次会慢一些，因为要加载网络并编译 CUDA 内核。

Locally one main search (220 visits) takes about 14 seconds and a forced evaluation of a teacher move (100 visits) about 4.5 seconds (from the local progress logs). The trial also checks the whole chain: KataGo starts, the perspective self test says `BLACK (selftest)`, and the output passes `treesync.check_analysis`. The first run is slower because the network is loaded and CUDA kernels are compiled.

In [ ]:
t0 = time.time()
rc = run_stream([sys.executable, '-u', 'analyze_remote.py', '--in', min(inp, key=os.path.getsize), '--out', WORK + '/trial', '--katago', ENV['katago'], '--model', MODEL,
                 '--expect-sha256', EXPECTED_SHA256, '--limit', '3', '--force'])
print('trial exit code %d, %.0f seconds (the first run includes loading the network and compiling kernels)' % (rc, time.time() - t0))

## 5. 完整批量运行（可随时中断，重新运行会续跑） / Full batch run (can be interrupted at any time, re-running resumes)

按 zip 里 manifest 给出的顺序逐组分析（从本机最后才会处理的分组开始）。每组结束后打印用时和每个查询的秒数；也可以随时打开输出目录里的 `g_<名称>.a2.meta.json` 看真实速度。

The groups are analysed in the order of the manifest in the zip (starting with the groups the local run reaches last). After each group the time and the seconds per query are printed; you can also open `g_<name>.a2.meta.json` in the output folder at any time to read the real speed.

In [ ]:
rc = run_stream([sys.executable, '-u', 'analyze_remote.py', '--in', IN_DIR, '--out', OUT_DIR, '--katago', ENV['katago'], '--model', MODEL,
                 '--expect-sha256', EXPECTED_SHA256, '--visits', str(VISITS), '--forced', str(FORCED)])
print('exit code', rc, '(0 = every group fine; 4 = the engine stopped, run this cell again to continue)')

## 6. 打包结果 / Package the results

只有通过 `treesync.check_analysis`（树指纹、行棋方视角、完整标记）的 `a2.json` 才会被打进 `a2_results.zip`。Colab 会自动下载；Kaggle 在右侧 *Output* 里下载；使用 Drive 时 zip 也会复制到 `MyDrive/josekiapp/`。

Only `a2.json` files that pass `treesync.check_analysis` (tree fingerprint, side-to-move perspective, complete flag) go into `a2_results.zip`. Colab downloads it automatically; on Kaggle download it from the *Output* panel on the right; with Drive the zip is also copied to `MyDrive/josekiapp/`.

In [ ]:
import treesync as ts
rows, good = [], []
for f in sorted(glob.glob(IN_DIR + '/g_*.n.json')):
    name = os.path.basename(f)[:-len('.n.json')]
    a2 = os.path.join(OUT_DIR, name + '.a2.json')
    if not os.path.exists(a2):
        rows.append((name, 'NOT DONE', '', ''))
        continue
    N = json.load(open(f, encoding='utf-8-sig'))
    A = json.load(open(a2, encoding='utf-8-sig'))
    st, msg = ts.check_analysis(N, A)
    meta = {}
    if os.path.exists(a2[:-5] + '.meta.json'):
        meta = json.load(open(a2[:-5] + '.meta.json', encoding='utf-8'))
    rows.append((name, st, meta.get('seconds', ''), meta.get('sec_per_query', '')))
    if st == 'ok':
        good.append(a2)
        if os.path.exists(a2[:-5] + '.meta.json'):
            good.append(a2[:-5] + '.meta.json')
print('%-44s %-12s %8s %12s' % ('group', 'status', 'seconds', 's/query'))
for r in rows:
    print('%-44s %-12s %8s %12s' % r)
info = ['packaged %s' % time.strftime('%Y-%m-%d %H:%M:%S'), 'weights sha256 %s' % sha256_of(MODEL), 'katago ' + sh('%s version | head -3' % ENV['katago']).replace('\n', ' | '), GPU]
open(WORK + '/run_info.txt', 'w', encoding='utf-8').write('\n'.join(info) + '\n')
zp = os.path.join(WORK, 'a2_results.zip')
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in good:
        z.write(f, os.path.basename(f))
    for extra in (OUT_DIR + '/run_summary.json', WORK + '/run_info.txt'):
        if os.path.exists(extra):
            z.write(extra, os.path.basename(extra))
print('%d analysis files -> %s (%d bytes)' % (sum(1 for g in good if g.endswith('.a2.json')), zp, os.path.getsize(zp)))
if DRIVE_BASE:
    shutil.copy(zp, os.path.join(DRIVE_BASE, 'a2_results.zip'))
    print('copy on Drive:', os.path.join(DRIVE_BASE, 'a2_results.zip'))
if ON_COLAB:
    from google.colab import files
    files.download(zp)
elif ON_KAGGLE:
    print('Kaggle: download a2_results.zip from the Output panel on the right (or from the saved version after Save & Run All).')

## 7. 之后 / Afterwards

下载 `a2_results.zip`，按 README_cloud.md 里的命令把 `g_*.a2.json` 放进 `D:\tmp\work`（已有的 `a2.json` 不要覆盖），然后对每个分组运行 `wq.py check <分组名>`，状态必须是 `analysis: ok`。没有完成的分组留给本机继续，或者重新运行第 5 步单元格。

Download `a2_results.zip`, put the `g_*.a2.json` files into `D:\tmp\work` with the commands in README_cloud.md (never overwrite an existing `a2.json`), then run `wq.py check <group name>` for each group; the status must be `analysis: ok`. Groups that are not done are left to the local run, or run the cell of step 5 again.